## upgrade molecules 

In [2]:

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem import rdChemReactions
from rdkit.Chem.MolStandardize import rdMolStandardize

def num_hdonors(mol):
    return rdMolDescriptors.CalcNumHBD(mol)

def sanitize(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None

# 1) Tautomerize and pick the one with max HBD (neutral transformation)
taut_enum = rdMolStandardize.TautomerEnumerator()
def pick_tautomer_max_hbd(mol):
    try:
        tautomers = taut_enum.Enumerate(mol)
    except Exception:
        return mol, num_hdonors(mol), 'orig'
    best = mol
    best_hbd = num_hdonors(mol)
    tag = 'orig'
    for t in tautomers:
        hbd = num_hdonors(t)
        if hbd > best_hbd:
            best, best_hbd = t, hbd
            tag = 'taut'
    return best, best_hbd, tag

import random
def upgrade_donor_poor_smiles(smiles_list, target_fraction=1.0, max_delta_atoms=1, max_new_mw=1):
    """
    - Only attempts to modify donor-poor molecules (HBD==0).
    - Works on a random subset (target_fraction) to avoid over-shifting the distribution.
    - Drops modifications that grow too much (atoms or MW).
    """
    out = []
    cont_total = 0
    cont_tau = 0
    for smi in tqdm(smiles_list):
        mol0 = sanitize(smi)
        if mol0 is None:
            out.append(smi)
            continue
        hbd_0 = num_hdonors(mol0)
        if num_hdonors(mol0) > 3 or random.random() > target_fraction:
            out.append(smi)
            continue

        mol_new, hbd_new, tag = pick_tautomer_max_hbd(mol0)
        smi_new = Chem.MolToSmiles(mol_new)
        mol_new = sanitize(smi_new)
        if mol_new is None:
            out.append(smi)
            continue

        cont_total += 1
        # Conservative guards
        if tag == 'taut':
            cont_tau += 1
        #print(tag, hbd_0, hbd_new)
        if (mol_new.GetNumAtoms() - mol0.GetNumAtoms()) > max_delta_atoms:
            print(f"Too many atoms: {mol_new.GetNumAtoms() - mol0.GetNumAtoms()}")
            #out.append(smi); continue
        if (Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)) > max_new_mw:
            print(f"Too much weight: {Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)}")
            #out.append(smi); continue

        out.append(smi_new)
    return out, cont_total, cont_tau

In [3]:
# Utilities: score all tautomers of a SMILES with the time model and pick the best
import os, sys
import torch
import numpy as np
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from torch_geometric.data import Data

# Ensure project root is on sys.path so we can import project modules
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("."), "..")))

from lib_functions.models import GINETimePredictor_MorganFP

# Local helpers from this folder
from lib_functions.data_preparation_utils import mol_to_graph
from lib_functions.data_preparation_utils import compute_features_cero_fps
# Time model class


def _build_time_model_input_from_mol(mol: Chem.Mol, device: torch.device) -> Data:
    """Build the PyG Data object for the time model from an RDKit Mol.
    Uses the same feature pipeline as in sample_molecules_FPSmodel.py via compute_features_cero_fps.
    """
    g = mol_to_graph(mol)
    if g is None:
        return None
    # Compute all features (graph, node, distances, edge index/attr, DOSD, fingerprint)
    ruido, gemb, nemb, distances, edge_index, edge_attr, natoms, _num, dosd_positions, fingerprint = compute_features_cero_fps(g)

    # Convert to tensors on device
    x = nemb.to(device)
    xA = gemb.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    distances = torch.tensor(distances, device=device, dtype=torch.float32)
    dosd_distances = torch.tensor(dosd_positions, device=device, dtype=torch.float32)
    morgan_fp = torch.tensor(fingerprint, device=device, dtype=torch.float32).unsqueeze(0)

    # Single-graph batch vector
    batch_vec = torch.zeros(x.size(0), dtype=torch.long, device=device)

    d = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        xA=xA,
        distances=distances,
        dosd_distances=dosd_distances,
        morgan_fp=morgan_fp,
        batch=batch_vec,
    )
    return d


def load_time_model(checkpoint_path: str, device: torch.device = None) -> GINETimePredictor_MorganFP:
    """Load the time model with weights and set to eval mode."""
    model = GINETimePredictor_MorganFP()
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


def score_mol_time(mol: Chem.Mol, time_model: GINETimePredictor_MorganFP) -> float:
    """Return the scalar time prediction for a single RDKit Mol using the time model."""
    device = next(time_model.parameters()).device
    try:
        data = _build_time_model_input_from_mol(mol, device)
        if data is None:
            return float('inf')
        with torch.no_grad():
            pred = time_model(data)
            # pred shape [1, 1]; detach to CPU scalar
            return float(pred.squeeze().detach().cpu().item())
    except Exception:
        return float('inf')


def pick_tautomer_min_time(mol: str, time_model: GINETimePredictor_MorganFP):
    """Enumerate tautomers and pick the one with minimum predicted time.
    Returns (best_smiles, best_time, num_tautomers).
    """
    taut_enum = rdMolStandardize.TautomerEnumerator()
    try:
        tautomers = list(taut_enum.Enumerate(mol))
    except Exception:
        tautomers = [mol]

    if not tautomers:
        tautomers = [mol]

    orig_smi = Chem.MolToSmiles(mol, isomericSmiles=True)
    best_mol = mol
    best_score = score_mol_time(mol, time_model)
    tag = 'orig'
    orig_hbd = num_hdonors(mol)
    for t in tautomers:
        num_hbd = num_hdonors(t)
        if orig_hbd >= num_hbd:
            continue    
        # Ensure valid & sanitized
        smi_t = Chem.MolToSmiles(t, isomericSmiles=True)
        mt = Chem.MolFromSmiles(smi_t)
        if mt is None:
            continue
        try:
            Chem.SanitizeMol(mt)
        except Exception:
            continue
        score = score_mol_time(mt, time_model)
        if score < best_score and smi_t != orig_smi:
            #print(score, best_score)
            #print(smi_t, Chem.MolToSmiles(best_mol, isomericSmiles=True))
            best_score = score
            best_mol = mt
            tag = 'taut'

    if best_mol is None:
        return smiles, float('inf'),  'orig'

    return Chem.MolToSmiles(best_mol, isomericSmiles=True), best_score,  tag


def pick_tautomer_min_time_then_hbd(smiles: str, time_model: GINETimePredictor_MorganFP, rdkit_calc_hbd):
    """Pick the min-time tautomer, then (optionally) prefer the one with higher HBD among equal-time ties.
    rdkit_calc_hbd: function like rdMolDescriptors.CalcNumHBD.
    """
    smi_best, score_best, _ = pick_tautomer_min_time(smiles, time_model)
    return smi_best, score_best

def upgrade_donor_poor_smiles_time(smiles_list, time_model, target_fraction=1.0, max_delta_atoms=1, max_new_mw=1):
    """
    - Only attempts to modify donor-poor molecules (HBD==0).
    - Works on a random subset (target_fraction) to avoid over-shifting the distribution.
    - Drops modifications that grow too much (atoms or MW).
    """
    # Load your time model once
    
    out = []
    cont_total = 0
    cont_tau = 0
    hbd_diff = []
    for smi in tqdm(smiles_list):
        mol0 = sanitize(smi)
        if mol0 is None:
            out.append(smi)
            hbd_diff.append(0)
            continue
        hbd_0 = num_hdonors(mol0)
        if num_hdonors(mol0) > 3 or random.random() > target_fraction:
            out.append(smi)
            hbd_diff.append(0)
            continue

        smi_new, best_score, tag = pick_tautomer_min_time(mol0, time_model)
        mol_new = sanitize(smi_new)
        hbd_new = num_hdonors(mol_new)
        if mol_new is None:
            out.append(smi)
            hbd_diff.append(0)
            continue

        cont_total += 1
        # Conservative guards
        if tag == 'taut':
            cont_tau += 1
            hbd_diff.append(hbd_new - hbd_0)
        #print(tag, hbd_0, hbd_new)
        if (mol_new.GetNumAtoms() - mol0.GetNumAtoms()) > max_delta_atoms:
            print(f"Too many atoms: {mol_new.GetNumAtoms() - mol0.GetNumAtoms()}")
            #out.append(smi); continue
        if (Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)) > max_new_mw:
            print(f"Too much weight: {Descriptors.MolWt(mol_new) - Descriptors.MolWt(mol0)}")
            #out.append(smi); continue

        out.append(smi_new)
    return out, cont_total, cont_tau, hbd_diff



Running on cuda
Running on cuda


## Processing of PubChem data

In [4]:
import tqdm

In [1]:
# read the file ../Data/CID-SMILES 
import tqdm 
with open('../Data/CID-SMILES-filtered-lt70-notraining.txt', 'r') as f:
    contador = 0
    smiles_list = []
    for line in tqdm.tqdm(f):
        smiles_list.append(line)
        contador += 1

print(len(smiles_list))

94697885it [00:43, 2182638.09it/s]

94697885


In [2]:
# make 5 random samples of 1M smiles in CID-SMILES-filtered-lt70_x.txt
import random
for i in range(5):
    with open(f'../Data/CID-SMILES-filtered-lt70_{i}_notraining.txt', 'w') as f:
        for smile in random.sample(smiles_list, 1000000):
            f.write(smile)


In [ ]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/250209_database_allmolecules_main_2_22_sinfps_timepred_2_22_sinfps_sinexplicit/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997700
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9977}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.949521
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.10168277968401368, 'MolLogP': 0.022153917901986026, 'MolWt': 0.05336217669126425, 'TPSA': 0.00901716806989947, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9978}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.959506
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.10052073234259877, 'MolLogP': 0.02422561902770322, 'MolWt': 0.052323320531104736, 'TPSA': 0.008154418070536514, 'NumHAcc

...

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9976}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.959091
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09703970041705838, 'MolLogP': 0.02598411199726456, 'MolWt': 0.05037145105496728, 'TPSA': 0.011380870306846277, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9983}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998000
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9980}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.957945
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09798951239593179, 'MolLogP': 0.028173392224997082, 'MolWt': 0.051357718589535514, 'TPSA': 0.008019544802537346, 'NumHAc

: 

In [1]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998300
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9983}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.956377
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09447173732065166, 'MolLogP': 0.02038362020842204, 'MolWt': 0.053265954404096774, 'TPSA': 0.010050005205966609, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998500
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9985}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964087
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09385036309457248, 'MolLogP': 0.022800802501922825, 'MolWt': 0.052298058311098905, 'TPSA': 0.008573625098308677, 'NumHAc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998400
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9984}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964799
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08779934953829943, 'MolLogP': 0.026030825461585305, 'MolWt': 0.04752248992192998, 'TPSA': 0.00821026323389536, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998100
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9981}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964488
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0894793189891889, 'MolLogP': 0.024448582895301375, 'MolWt': 0.050274947161870594, 'TPSA': 0.01077412311862261, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999000
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9990}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.963573
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09107069631062777, 'MolLogP': 0.02695562982007913, 'MolWt': 0.051297241682671746, 'TPSA': 0.007850284986057797, 'NumHAcc

In [11]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_old.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.965727
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08768305187104429, 'MolLogP': 0.025320211518836545, 'MolWt': 0.047287987575331675, 'TPSA': 0.007203539729819245, 'NumHAc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997900
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9979}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964632
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0871572267369794, 'MolLogP': 0.024576902357118943, 'MolWt': 0.046325089214649826, 'TPSA': 0.007353222251063388, 'NumHAcc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998100
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9981}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.965887
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08123077782967118, 'MolLogP': 0.023590999020049054, 'MolWt': 0.04453149343726291, 'TPSA': 0.008476382543551833, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.957254
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08359064899716137, 'MolLogP': 0.021245478855415147, 'MolWt': 0.04762436526875379, 'TPSA': 0.009256511348945432, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_old.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9988}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.964617
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09035955647544924, 'MolLogP': 0.020104574619869656, 'MolWt': 0.044131624634868506, 'TPSA': 0.007098068064943521, 'NumHAc

In [2]:
from typing import List
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/defog_final_smiles_strict.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 20000/20000 [00:01<00:00, 11939.95it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902300
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9023}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.935528
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.19039273197643009, 'MolLogP': 0.02260089962171215, 'MolWt': 0.1530179271835068, 'TPSA': 0.07487142695220401, 'NumHAccepto

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9027}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.934909
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.19016800509919285, 'MolLogP': 0.027293963606306013, 'MolWt': 0.1490253138962689, 'TPSA': 0.07493027638733472, 'NumHAccept

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9028}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.938063
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18128182592389971, 'MolLogP': 0.02850240451817975, 'MolWt': 0.14090971696485233, 'TPSA': 0.07096788421813355, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9028}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.934684
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18559633548987592, 'MolLogP': 0.027744876346936787, 'MolWt': 0.14940090034267414, 'TPSA': 0.07966893145327543, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9036}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.903600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9036}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.902600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9026}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.935236
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1901634490769083, 'MolLogP': 0.034697872116097817, 'MolWt': 0.14740045335294163, 'TPSA': 0.07069894486788585, 'NumHAccept

In [3]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger

class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/defog_final_smiles_relaxed.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 20000/20000 [00:01<00:00, 10622.67it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9832}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.934415
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18652698860908237, 'MolLogP': 0.022389544664146548, 'MolWt': 0.13661555556622668, 'TPSA': 0.09001608594693605, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983600
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9836}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.933127
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18645346572438928, 'MolLogP': 0.02654807353754425, 'MolWt': 0.1330233325557184, 'TPSA': 0.08961120643146205, 'NumHAccepto

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983800
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9838}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.937305
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.17706344306776942, 'MolLogP': 0.027613918871804873, 'MolWt': 0.12487748474122519, 'TPSA': 0.08562153481472826, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9837}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.933008
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18126695007012164, 'MolLogP': 0.026928819477547217, 'MolWt': 0.13303014849736516, 'TPSA': 0.09459726468682925, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9846}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.984600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9846}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9837}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.934061
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.18636670368183073, 'MolLogP': 0.03401431756271047, 'MolWt': 0.13147967310885447, 'TPSA': 0.08620818337961543, 'NumHAccept

In [4]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/digress_generated_all.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



  0%|          | 0/18000 [00:00<?, ?it/s]

100%|██████████| 18000/18000 [00:02<00:00, 7970.87it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833300
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8333}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.911851
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.27589327686971427, 'MolLogP': 0.022735494446412326, 'MolWt': 0.12653707779478124, 'TPSA': 0.07483286703778085, 'NumHAccep

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8334}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.914442
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.2633188896980571, 'MolLogP': 0.028779823223060972, 'MolWt': 0.11347051597184878, 'TPSA': 0.07169798394054534, 'NumHAccept

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833400
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8334}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.913807
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.271639793317696, 'MolLogP': 0.029380741722775182, 'MolWt': 0.1214227777316997, 'TPSA': 0.08184189916894205, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8337}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.833700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 8337}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.833600
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 8336}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.914678
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.2710355110499318, 'MolLogP': 0.030494850624543657, 'MolWt': 0.12298267111847533, 'TPSA': 0.07112730397685225, 'NumHAccept

In [1]:
from typing import List
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/gdss_zinc250k-sample.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            smiles = parts[0]
            validity = int(parts[1])
            if validity == 0:  # valid molecule
                smiles_list.append(smiles)
            else:  # invalid molecule
                smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 10000/10000 [00:00<00:00, 11649.55it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.944200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9442}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9414}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.700080
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5559441442368517, 'MolLogP': 0.06343003821557469, 'MolWt': 0.8336371977617747, 'TPSA': 0.44488751064113363, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.944200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9442}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941300
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9413}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.696808
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5665923325557002, 'MolLogP': 0.06475450839909346, 'MolWt': 0.8653291656619598, 'TPSA': 0.4561581661634979, 'NumHAcceptors

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.944200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9442}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941500
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9415}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.696056
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5670653420309952, 'MolLogP': 0.0631362526281069, 'MolWt': 0.8291148554946373, 'TPSA': 0.45601282211131927, 'NumHAcceptors

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.944200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9442}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9418}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.703027
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5584611577822861, 'MolLogP': 0.06593849587398538, 'MolWt': 0.7963364918853514, 'TPSA': 0.44848999064198003, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.944200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9442}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.942600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9426}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.941500
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9415}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.692296
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.5728632596728536, 'MolLogP': 0.07016884021104902, 'MolWt': 0.8466756872591215, 'TPSA': 0.45301030613800614, 'NumHAcceptor

In [2]:
from typing import List
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger
from tqdm import tqdm
from rdkit import Chem
import random

class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/GRUM_zinc250k.txt', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            smiles = parts[0]
            validity = int(parts[1])
            if validity == 0:  # valid molecule
                smiles_list.append(smiles)
            else:  # invalid molecule
                smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")
# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [00:01<00:00, 8871.50it/s]
INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9842}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.982900
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9829}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.868530
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.1372715346571223, 'MolLogP': 0.39496578670634214, 'MolWt': 0.35975218296862754, 'TPSA': 0.13495784734701655, 'NumHAccepto

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9842}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.982900
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9829}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.867884
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.13979424863853446, 'MolLogP': 0.3914448972142929, 'MolWt': 0.3575301238985314, 'TPSA': 0.1360297554670742, 'NumHAcceptors

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9842}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.982800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9828}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.865428
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.13969535032534972, 'MolLogP': 0.4206604822485565, 'MolWt': 0.33454732389868885, 'TPSA': 0.1320625834309414, 'NumHAcceptor

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.984200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 9842}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.983700
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9837}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.983000
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9830}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.861898
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.14981756331092172, 'MolLogP': 0.4173498697927926, 'MolWt': 0.3716152103872065, 'TPSA': 0.13119028965147517, 'NumHAcceptor

In [ ]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger
from tqdm import tqdm
from rdkit import Chem
import random


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/CDMOL_guacamol_smiles.smiles', 'r') as f:
    lines = f.readlines()
    smiles_list = []
    for line in lines:
        smiles_list.append(line.strip())

if len(smiles_list) <10000:
   for i in range(10000-len(smiles_list)):
      smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")

if len(filtered_smiles) <10000:
    # randomly duplicate some smiles to reach 10000
    for i in range(10000-len(filtered_smiles)):
        filtered_smiles.append(random.choice(filtered_smiles))

print(len(filtered_smiles))

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 10000/10000 [00:00<00:00, 11864.55it/s]
INFO : Benchmarking distribution learning, version v2


10000
Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.751100
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7511}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.954602
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0645296057774216, 'MolLogP': 0.018110347258852157, 'MolWt': 0.058213703122524066, 'TPSA': 0.009800515806120225, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7506}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.953490
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0727838117864628, 'MolLogP': 0.021453153502832306, 'MolWt': 0.06360387047859188, 'TPSA': 0.0107538432071926, 'NumHAccepto

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7506}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.954554
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.07098576301780021, 'MolLogP': 0.025978802693960603, 'MolWt': 0.06841757611217736, 'TPSA': 0.012319110961034973, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7502}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.952493
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.06753700795574506, 'MolLogP': 0.023857996696075523, 'MolWt': 0.06012689008254953, 'TPSA': 0.012840464585161132, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 0.848200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 8482}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.751200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 7512}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.750800
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 7508}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.952089
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.07477862317027886, 'MolLogP': 0.02551667934319582, 'MolWt': 0.06493923598569591, 'TPSA': 0.013270252933356819, 'NumHAccep

In [3]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger
from tqdm import tqdm
from rdkit import Chem
import random


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]


# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/jtvae_generated_all_231108.txt', 'r') as f:
    smiles_list = [line.strip() for line in f]


if len(smiles_list) <10000:
   for i in range(10000-len(smiles_list)):
      smiles_list.append("None")

# Filter SMILES with less than 70 atoms (including implicit H)
filtered_smiles = []
atom_counts = []
for smiles in tqdm(smiles_list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:  # Check if SMILES is valid
        # Add explicit hydrogens to count all atoms including implicit H
        mol_with_h = Chem.AddHs(mol)
        num_atoms = mol_with_h.GetNumAtoms()
        atom_counts.append(num_atoms)
        if num_atoms < 70:
            filtered_smiles.append(smiles)
    else:
        filtered_smiles.append("None")

if len(filtered_smiles) <10000:
    # randomly duplicate some smiles to reach 10000
    for i in range(10000-len(filtered_smiles)):
        filtered_smiles.append(random.choice(filtered_smiles))

print(len(filtered_smiles))

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=filtered_smiles)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



100%|██████████| 30000/30000 [00:03<00:00, 9647.77it/s] 
INFO : Benchmarking distribution learning, version v2


30000
Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9976}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.565527
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7634328680682771, 'MolLogP': 1.2372384666873448, 'MolWt': 5.009157468972942, 'TPSA': 0.3087045134915995, 'NumHAcceptors'

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998000
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9980}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.547312
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.8592120166467438, 'MolLogP': 1.2133908881496476, 'MolWt': 5.057271367962629, 'TPSA': 0.3259261732527775, 'NumHAcceptors'

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997600
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9976}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.559786
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7192660594759717, 'MolLogP': 1.2176550578445218, 'MolWt': 4.959735128892996, 'TPSA': 0.3060703866005193, 'NumHAcceptors'

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997100
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9971}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.557152
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.744488249790515, 'MolLogP': 1.2678765233103968, 'MolWt': 5.096547400904113, 'TPSA': 0.3206353444188583, 'NumHAcceptors':

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.996400
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9964}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.546493
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 1.7733775928981461, 'MolLogP': 1.2445896037198507, 'MolWt': 5.1232377077598406, 'TPSA': 0.2899854051346338, 'NumHAcceptors

# Time model hablation

In [4]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/251010_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit_notime/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9992}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.934470
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08987493707195678, 'MolLogP': 0.007591869969341428, 'MolWt': 0.05147977645124469, 'TPSA': 0.007401083497966265, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999000
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9990}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.946176
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08965310757523368, 'MolLogP': 0.00889799092150937, 'MolWt': 0.051223911294192195, 'TPSA': 0.007114969342528022, 'NumHAcc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999100
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9991}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.947781
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08196900674009333, 'MolLogP': 0.009091363700223558, 'MolWt': 0.04620963011533967, 'TPSA': 0.007092997219083281, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9988}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.946415
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08479643354277852, 'MolLogP': 0.010588639878703458, 'MolWt': 0.04874056485914472, 'TPSA': 0.009916926137320667, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9996}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999400
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9994}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.945491
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0871187726338704, 'MolLogP': 0.009972907341181378, 'MolWt': 0.04964046569824025, 'TPSA': 0.006341437277640154, 'NumHAcce

In [1]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from tqdm import tqdm
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

# Source molecules for the generator
smiles_csv = '../mols_gen/251012_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit_unseed2/all_generated_molecules.csv'
smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_notraining.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998100
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9981}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.945480
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0919085478959549, 'MolLogP': 0.017584754440573218, 'MolWt': 0.05231545811270015, 'TPSA': 0.007626916178407781, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998000
INFO :   Sampling time: 0:00:03
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9980}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.954925
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09218662083036765, 'MolLogP': 0.020722796098570556, 'MolWt': 0.05338137994255614, 'TPSA': 0.008644341258916074, 'NumHAcc

...

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997800
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9978}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.955058
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08880990319553225, 'MolLogP': 0.024459722689944063, 'MolWt': 0.04768953449525682, 'TPSA': 0.009901845067220695, 'NumHAcc

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_notraining.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.998600
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9986}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.955029
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09185583574799265, 'MolLogP': 0.01893204694013436, 'MolWt': 0.05065318501576695, 'TPSA': 0.0055267337145122365, 'NumHAcc

: 

# Novelty whole PubChem

In [ ]:
# read the file ../Data/CID-SMILES 
import tqdm 
with open('../Data/CID-SMILES-filtered-lt70.txt', 'r') as f:
    contador = 0
    smiles_list_pc = []
    for line in tqdm.tqdm(f):
        smiles_list_pc.append(line)
        contador += 1

smiles_csv = '../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv'
smiles_list_gen = pd.read_csv(smiles_csv).smiles.to_list()
